In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.lightgbm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier
import sys; sys.path.append('..')
from src.features.feature_engineering import CreditRiskFeatures

# Load data
df = pd.read_csv('../data/raw/application_train.csv')
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print(f'Features: {X.shape[1]}')
print(f'Samples: {X.shape[0]:,}')
print(f'Class balance: {y.value_counts().to_dict()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)


Features: 120
Samples: 307,511
Class balance: {0: 282686, 1: 24825}


In [4]:
mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('Credit-risk-default-prediction')

# Build sklearn Pipeline
pipeline = Pipeline([
    ('features', CreditRiskFeatures()),
    ('model', LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.7,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    ))
])

with mlflow.start_run(run_name='lgbm-baseline-v1'):
    # Log parameters
    mlflow.log_params(pipeline.named_steps['model'].get_params())

    # Train
    pipeline.fit(X_train, y_train)

    # Evaluate
    preds_proba = pipeline.predict_proba(X_test)[:, 1]
    preds_class = pipeline.predict(X_test)
    auc = roc_auc_score(y_test, preds_proba)

    # Gini coefficient (used in banking)
    gini = 2 * auc - 1

    # Log metrics
    mlflow.log_metric('test_auc', auc)
    mlflow.log_metric('gini', gini)

    # Log model to registry
    mlflow.sklearn.log_model(
        pipeline,
        'credit-risk-pipeline',
        registered_model_name='credit-risk-lgbm'
    )

    print(f'AUC: {auc:.4f}  |  Gini: {gini:.4f}')
    print(classification_report(y_test, preds_class))


2026/03/05 20:25:25 INFO mlflow.tracking.fluent: Experiment with name 'Credit-risk-default-prediction' does not exist. Creating a new experiment.
c:\Users\Shanu\Desktop\Final_projects\fraud_detection\credit-risk-platform\venv\lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Shanu\Desktop\Final_projects\fraud_detection\credit-risk-platform\venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\Shanu\Desktop\Final_projects\fraud_detection\credit-risk-platform\venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_p

AUC: 0.7691  |  Gini: 0.5381
              precision    recall  f1-score   support

           0       0.96      0.76      0.85     56538
           1       0.19      0.63      0.29      4965

    accuracy                           0.75     61503
   macro avg       0.57      0.70      0.57     61503
weighted avg       0.90      0.75      0.81     61503

🏃 View run lgbm-baseline-v1 at: http://localhost:5000/#/experiments/354193471891864400/runs/76a20b492ed14387b334ef1ad5fa491b
🧪 View experiment at: http://localhost:5000/#/experiments/354193471891864400
